<a href="https://colab.research.google.com/github/HrishikeshWadile/SoftwarePractice/blob/main/RNN_and_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets --quiet

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
def tokenize(text):
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text.lower().split()

counter = Counter()

for sample in train_data:
    counter.update(tokenize(sample["text"]))

vocab = {"<pad>": 0, "<unk>": 1}

for word, freq in counter.items():
    if freq > 5:
        vocab[word] = len(vocab)

vocab_size = len(vocab)
print("Vocab Size:", vocab_size)

Vocab Size: 26208


In [5]:
class IMDBDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]["text"]
        label = self.data[idx]["label"]

        tokens = tokenize(text)
        indices = [vocab.get(word, vocab["<unk>"]) for word in tokens]

        return torch.tensor(indices), torch.tensor(label, dtype=torch.float32)

In [6]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    texts = [item[0] for item in batch]
    labels = torch.stack([item[1] for item in batch])

    lengths = torch.tensor([len(t) for t in texts])

    texts = pad_sequence(texts, batch_first=True)

    return texts.to(device), lengths.to(device), labels.to(device)

In [7]:
train_loader = DataLoader(
    IMDBDataset(train_data),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    IMDBDataset(test_data),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

In [8]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_output, hidden = self.rnn(packed)

        out = self.fc(hidden[-1])
        return out.squeeze()

In [9]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(packed)

        out = self.fc(hidden[-1])
        return out.squeeze()

In [10]:
def train_model(model, loader, epochs=3):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for texts, lengths, labels in loader:
            optimizer.zero_grad()

            outputs = model(texts, lengths)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}, Accuracy: {correct/total:.4f}")

In [11]:
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for texts, lengths, labels in loader:
            outputs = model(texts, lengths)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print("Test Accuracy:", correct/total)

In [12]:
rnn_model = RNNModel(vocab_size, 100, 128).to(device)

print("Training RNN...")
train_model(rnn_model, train_loader, epochs=3)

evaluate_model(rnn_model, test_loader)

Training RNN...
Epoch 1, Loss: 0.6694, Accuracy: 0.5890
Epoch 2, Loss: 0.6433, Accuracy: 0.6284
Epoch 3, Loss: 0.6460, Accuracy: 0.6147
Test Accuracy: 0.58588


In [13]:
lstm_model = LSTMModel(vocab_size, 100, 128).to(device)

print("Training LSTM...")
train_model(lstm_model, train_loader, epochs=3)

evaluate_model(lstm_model, test_loader)

Training LSTM...
Epoch 1, Loss: 0.6780, Accuracy: 0.5738
Epoch 2, Loss: 0.6448, Accuracy: 0.6247
Epoch 3, Loss: 0.5064, Accuracy: 0.7539
Test Accuracy: 0.76252
